In [6]:
# Cell 1: undetected-chromedriver 설치
# undetected-chromedriver와 selenium을 설치합니다.
%pip install undetected-chromedriver selenium


Note: you may need to restart the kernel to use updated packages.


In [40]:
# Cell 2: 네이버 쇼핑 키워드 기반 상품 검색 크롤링
# 검색어를 입력받아 네이버 쇼핑에서 최대 30개의 상품 정보를 수집합니다.
import sys
import tempfile
import subprocess
import os
import json
import re
from urllib.parse import quote

# 노트북에서 시각 디버깅을 원하면 False로 설정하세요
NOTEBOOK_HEADLESS = False

# 검색할 키워드 입력 (여기를 수정하세요)
SEARCH_KEYWORD = "가방"  # 예시: "노트북", "스마트폰", "명품가방" 등

# 최대 수집할 상품 개수
MAX_PRODUCTS = 30

# 검색 URL 생성
search_url = f"https://search.shopping.naver.com/ns/search?query={quote(SEARCH_KEYWORD)}"

print(f"🔍 검색 키워드: {SEARCH_KEYWORD}")
print(f"🔗 검색 URL: {search_url}")
print(f"📦 최대 수집 개수: {MAX_PRODUCTS}개\n")

# 크롤링 스크립트 생성
crawl_script = f"""import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import json
import time
import re

headless = {NOTEBOOK_HEADLESS!r}
search_url = {search_url!r}
max_products = {MAX_PRODUCTS}

products = []

bad_domains = ["help.pay.naver.com", "nid.naver.com"]

def extract_url_from_text(text: str) -> str:
    if not text:
        return ""
    match = re.search(r"(https?://[^\s]+)", text)
    return match.group(1).strip() if match else ""

def normalize_product_url(raw_url: str) -> str:
    if not raw_url:
        return ""
    candidate = raw_url.strip()
    if not candidate or candidate.startswith("javascript"):
        return ""
    if "http" not in candidate:
        embedded = extract_url_from_text(candidate)
        if embedded:
            candidate = embedded
    if candidate.startswith("//"):
        candidate = "https:" + candidate
    elif candidate.startswith("/"):
        candidate = "https://shopping.naver.com" + candidate
    if any(domain in candidate for domain in bad_domains):
        return ""
    return candidate if candidate.startswith("http") else ""

def normalize_image_url(raw_url: str) -> str:
    if not raw_url:
        return ""
    candidate = raw_url.strip().strip("'").strip('"')
    if not candidate or candidate.startswith("data:"):
        return ""
    if candidate.startswith("//"):
        candidate = "https:" + candidate
    elif candidate.startswith("/"):
        candidate = "https://shopping-phinf.pstatic.net" + candidate
    return candidate if candidate.startswith("http") else ""

def first_from_srcset(value: str) -> str:
    if not value:
        return ""
    for part in value.split(","):
        url_part = part.strip().split(" ")[0]
        if url_part:
            return url_part
    return ""

try:
    # undetected-chromedriver 설정 (봇 감지 우회)
    options = uc.ChromeOptions()
    
    if headless:
        options.add_argument('--headless=new')
    
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--no-sandbox')
    options.add_argument('--window-size=1920,1080')
    options.add_argument('--start-maximized')
    
    # User-Agent 설정
    options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')
    
    print("브라우저 시작 중...")
    driver = uc.Chrome(options=options, version_main=None)
    
    print(f"접속 중: {{search_url}}")
    
    # 먼저 네이버 메인 페이지로 접속
    print("네이버 메인 페이지 접속 중...")
    driver.get("https://www.naver.com")
    time.sleep(3)
    
    if not headless:
        time.sleep(5)
    
    # 검색 페이지로 이동
    print(f"\\n검색 페이지로 이동 중: {{search_url}}")
    driver.get(search_url)
    
    # 페이지 로드 대기
    print("페이지 로딩 대기 중...")
    if not headless:
        time.sleep(8)
    else:
        time.sleep(5)
    
    # 접속 제한 체크
    page_title = driver.title
    page_source = driver.page_source
    
    if "접속이 일시적으로 제한" in page_source or "Access Denied" in page_title:
        print("⚠ 접속 제한 페이지가 감지되었습니다.")
        if not headless:
            print("⚠ 브라우저 창에서 직접 새로고침을 시도해보세요.")
            time.sleep(15)  # 사용자가 처리할 시간 제공
            page_source = driver.page_source  # 다시 확인
    
    # 스크롤하여 동적 콘텐츠 로드 (더 많은 상품 로드)
    print("상품 로딩 중...")
    for scroll_idx in range(4):
        driver.execute_script("window.scrollBy(0, window.innerHeight * 0.5)")
        time.sleep(3)
    
    time.sleep(3)
    
    # 상품 리스트 찾기 (네이버 쇼핑 상품 카드)
    product_selectors = [
        "li[class*='productCardList_item']",  # 신 UI 상품 카드
        "div[class*='basicProductCardInformation']",  # 기본 상품 카드 정보
        "div[class*='productCardInfo']",  # 상품 카드 정보 영역
        "div[class*='productCard_wrap']",  # 상품 카드 래퍼
        "div[class*='productCardThumbnail']",  # 썸네일 카드 영역
        "div.adProduct_item_T7utB",  # (구) 광고 상품
        "div.adProduct_inner_xrvC_",  # (구) 광고 상품 내부 컨테이너
        "div.product_item_KQayS",  # (구) 일반 상품
        "div.superSavingProduct_item_6mR7_",  # (구) 특가 상품
        "div[class*='miniProductCardInformation']",  # (구) 카테고리 모듈 카드
        "div[class*='product_info_area']",  # (구) 상품 정보 영역
        "div[class*='product_info']",  # (구) 기타 상품 정보 영역
        "div[class*='product']"  # fallback
    ]
    
    product_items = []
    for selector in product_selectors:
        try:
            items = driver.find_elements(By.CSS_SELECTOR, selector)
            if items:
                product_items = items
                print(f"발견된 상품 수: {{len(product_items)}}개 (셀렉터: {{selector}})")
                break
        except Exception as selector_error:
            print(f"  셀렉터 '{{selector}}' 처리 중 오류: {{selector_error}}")
            continue
    
    if not product_items:
        print("⚠ 상품 리스트를 찾을 수 없습니다.")
        print(f"현재 URL: {{driver.current_url}}")
        print(f"페이지 제목: {{driver.title}}")
    else:
        # 각 상품 정보 추출
        for idx, item_elem in enumerate(product_items[:max_products]):
            try:
                if idx > 0:
                    time.sleep(0.3)
                
                card_scope = item_elem
                try:
                    card_scope = item_elem.find_element(By.XPATH, ".//ancestor::li[contains(@class,'productCardList_item')]")
                except Exception:
                    pass
                search_scope = card_scope or item_elem

                try:
                    driver.execute_script("arguments[0].scrollIntoView({{block: 'center', inline: 'nearest'}})", search_scope)
                    time.sleep(0.2)
                except Exception:
                    pass
                
                # 상품명 추출
                title = ""
                # 우선순위: 광고 상품 정보 영역의 링크 > 일반 상품 링크 > 기타
                title_selectors = [
                    "span[class*='basicProductCard_title']",
                    "strong[class*='basicProductCard_title']",
                    "div[class*='basicProductCardInformation'] strong[class*='title']",
                    "div[class*='basicProductCardInformation'] span[class*='title']",
                    "a[class*='basicProductCard_link'] strong",
                    "a[class*='basicProductCard_link'] span",
                    "a[class*='basicProductCard_link']",
                    "strong[class*='productCardTitle']",
                    "a[class*='productCardTitle']",
                    "div[class*='productCardInformation'] strong",
                    "span[class*='productTitle']",
                    "div.adProduct_title__fsQU6",
                    "div.adProduct_info_area_aHANq a",
                    "a.basicList_link__1MaTN",
                    "div.basicList_title__3P9Q7",
                    "div[class*='miniProductCardInformation'] a",
                    "span[class*='miniProductCardInformation_title']",
                    "div[class*='product_info_area'] a",
                    "div[class*='product_info'] a",
                    "a[class*='link']",
                    "a[class*='title']",
                ]
                
                disallowed_titles = {"디지털/가전", "패션", "뷰티", "식품", "생활", "가구", "도서", "스포츠", "완구", "반려동물"}
                warning_keywords = [
                    "출발",
                    "배송",
                    "멤버십",
                    "내일배송",
                    "오늘출발",
                    "무료반품",
                    "리뷰",
                    "할인",
                    "쿠폰",
                    "혜택",
                    "회원",
                    "N도착",
                    "N내일",
                ]

                def looks_like_bad_title(text: str) -> bool:
                    if not text:
                        return True
                    normalized = text.strip()
                    if len(normalized) < 4:
                        return True
                    lowered = normalized.lower()
                    for keyword in warning_keywords:
                        if keyword in normalized:
                            return True
                    if lowered.replace(" ", "").isdigit():
                        return True
                    if any(char.isdigit() for char in normalized) and all(ch.isdigit() or ch in ":./" for ch in normalized if not ch.isalpha()):
                        return True
                    return False
                
                for sel in title_selectors:
                    try:
                        title_elem = search_scope.find_element(By.CSS_SELECTOR, sel)
                        if title_elem:
                            title_text = title_elem.text.strip()
                            # 카테고리 패턴 제거 ("디지털/가전 > 노트북" 같은 것)
                            if ">" in title_text:
                                parts = title_text.split(">")
                                if len(parts) > 1:
                                    # ">" 앞부분이 카테고리인 경우, 뒷부분만 사용
                                    title_text = parts[-1].strip()
                            
                            if title_text and len(title_text) >= 4:
                                if title_text not in disallowed_titles and not looks_like_bad_title(title_text):
                                    title = title_text
                                    break
                    except Exception:
                        continue
                
                if not title:
                    try:
                        rich_links = search_scope.find_elements(By.CSS_SELECTOR, "a[class*='basicProductCard_link']")
                        for link in rich_links:
                            fallback = (
                                (link.get_attribute("aria-label") or "").strip()
                                or (link.get_attribute("title") or "").strip()
                                or (link.get_attribute("data-i18n-key") or "").strip()
                            )
                            if fallback and fallback not in disallowed_titles and not looks_like_bad_title(fallback):
                                title = fallback
                                break
                    except Exception:
                        pass
                
                # 여전히 제목이 없으면 모든 링크에서 찾기
                if not title:
                    try:
                        all_links = search_scope.find_elements(By.CSS_SELECTOR, "a")
                        for link in all_links:
                            link_text = link.text.strip()
                            if link_text and len(link_text) >= 4:
                                if ">" in link_text:
                                    parts = link_text.split(">")
                                    if len(parts) > 1:
                                        link_text = parts[-1].strip()
                                if link_text not in disallowed_titles and not looks_like_bad_title(link_text):
                                    title = link_text
                                    break
                    except:
                        pass
                
                if not title:
                    try:
                        block_text = (search_scope.text or "").strip()
                        if block_text:
                            first_line = block_text.split("\\n")[0].strip()
                            if first_line and first_line not in disallowed_titles and not looks_like_bad_title(first_line):
                                title = first_line
                    except Exception:
                        pass
                
                # 상품 링크 추출
                product_link = ""
                try:
                    link_selectors = [
                        "a[class*='productCardTitle']",
                        "a[class*='productCardThumbnail_link']",
                        "div[class*='basicProductCardInformation'] a",
                        "div[class*='productCardInformation'] a",
                        "div.adProduct_title__fsQU6 a",
                        "div.adProduct_info_area_aHANq a",
                        "div[class*='miniProductCardInformation'] a",
                        "a.thumbnail_thumb_MG0r2",
                        "a.basicList_link__1MaTN",
                        "div[class*='product_info_area'] a",
                        "div[class*='product_info'] a",
                        "a[class*='link']",
                        "a[href*='shopping.naver.com']",
                    ]

                    link_attr_candidates = [
                        "href",
                        "data-url",
                        "data-link",
                        "data-nclick",
                        "data-href",
                        "data-deeplink"
                    ]

                    for selector in link_selectors:
                        try:
                            link_elems = search_scope.find_elements(By.CSS_SELECTOR, selector)
                            if not link_elems:
                                continue
                            for link_elem in link_elems:
                                candidate_url = ""
                                for attr in link_attr_candidates:
                                    value = link_elem.get_attribute(attr)
                                    if value:
                                        if attr.endswith("srcset"):
                                            candidate_url = first_from_srcset(value)
                                        else:
                                            candidate_url = value.strip()
                                        if candidate_url:
                                            break
                                if not candidate_url:
                                    onclick = link_elem.get_attribute("onclick") or ""
                                    candidate_url = extract_url_from_text(onclick)
                                if not candidate_url:
                                    dataset_url = link_elem.get_attribute("data-criteo-clickurl") or ""
                                    candidate_url = dataset_url.strip()
                                normalized = normalize_product_url(candidate_url)
                                if normalized:
                                    product_link = normalized
                                    break
                            if product_link:
                                break
                        except Exception:
                            continue
                except Exception:
                    pass
                
                # 가격 정보 추출 (원가, 할인가)
                original_price = ""
                displayed_price = ""
                
                try:
                    price_containers = item_elem.find_elements(By.CSS_SELECTOR, "div[class*='productCardPrice'], div.adProduct_price_area__rkHze")
                    price_texts = []
                    for container in price_containers:
                        text = container.text.strip()
                        if text:
                            price_texts.append(text)
                    if not price_texts:
                        price_texts.append(item_elem.text)
                    
                    price_candidates = []
                    for text in price_texts:
                        price_patterns = re.findall(r'([\\d,]+)\\s*원', text)
                        for match in price_patterns:
                            candidate_raw = match
                            candidate = candidate_raw.replace(",", "")
                            if candidate and candidate.isdigit() and 4 <= len(candidate) <= 10:
                                if candidate not in [p.replace(",", "") for p in price_candidates]:
                                    price_candidates.append(candidate_raw)
                    
                    # 가격 정렬 및 할당
                    if price_candidates:
                        # 숫자로 변환하여 정렬 (오름차순)
                        price_nums = [(p.replace(",", ""), p) for p in price_candidates]
                        price_nums.sort(key=lambda x: int(x[0]))
                        
                        # 가장 작은 값이 할인가
                        displayed_price = price_nums[0][1] + "원"
                        # 두 번째 값이 있으면 원가 (더 큰 값)
                        if len(price_nums) > 1:
                            original_price = price_nums[-1][1] + "원"
                except Exception as e:
                    pass
                
                # 썸네일 이미지 URL 추출
                thumbnail_url = ""
                try:
                    img_selectors = [
                        "img[class*='productCardThumbnail_image']",
                        "div[class*='productCardThumbnail_thumbnail__KzO1N'] img",
                        "div[class*='productCardThumbnail'] img",
                        "div.adProduct_img_area_eubFn img",
                        "div[class*='img_area'] img",
                        "div[class*='miniProductCardThumbnail'] img",
                        "div[class*='thumbnail'] img",
                        "div[class*='product_img'] img",
                        "img[class*='product']",
                        "img"
                    ]
                    candidate_attrs = [
                        "src",
                        "data-src",
                        "data-lazy-src",
                        "data-lazy",
                        "data-original",
                        "data-img-src",
                        "data-image",
                        "data-image-src",
                        "data-srcset",
                        "srcset"
                    ]
                    for selector in img_selectors:
                        try:
                            img_elems = search_scope.find_elements(By.CSS_SELECTOR, selector)
                            for img_elem in img_elems:
                                candidate_url = ""
                                for attr in candidate_attrs:
                                    value = img_elem.get_attribute(attr)
                                    if not value:
                                        continue
                                    if attr.endswith("srcset"):
                                        candidate_url = first_from_srcset(value)
                                    else:
                                        candidate_url = value.strip()
                                    if candidate_url:
                                        break
                                candidate_url = normalize_image_url(candidate_url)
                                if candidate_url:
                                    thumbnail_url = candidate_url
                                    break
                            if thumbnail_url:
                                break
                        except Exception:
                            continue

                    if not thumbnail_url:
                        source_elems = search_scope.find_elements(By.CSS_SELECTOR, "source")
                        for source in source_elems:
                            srcset_val = source.get_attribute("srcset") or source.get_attribute("data-srcset")
                            candidate_url = normalize_image_url(first_from_srcset(srcset_val))
                            if candidate_url:
                                thumbnail_url = candidate_url
                                break

                    if not thumbnail_url:
                        bg_selectors = [
                            "div[class*='productCardThumbnail']",
                            "a[class*='productCardThumbnail']",
                            "div[class*='thumbnail']",
                            "a[class*='thumbnail']",
                            "span[class*='thumbnail']",
                            "div[class*='product_img']"
                        ]
                        for selector in bg_selectors:
                            try:
                                bg_elems = search_scope.find_elements(By.CSS_SELECTOR, selector)
                                for bg_elem in bg_elems:
                                    candidate_url = ""
                                    style_attr = bg_elem.get_attribute("style") or ""
                                    if style_attr:
                                        match = re.search(r"url\((.*?)\)", style_attr)
                                        if match:
                                            candidate_url = match.group(1).strip().strip("'").strip('"')
                                    if not candidate_url:
                                        data_lazy = bg_elem.get_attribute("data-lazy-background") or bg_elem.get_attribute("data-background-image") or ""
                                        candidate_url = data_lazy.strip().strip("'").strip('"')
                                    candidate_url = normalize_image_url(candidate_url)
                                    if candidate_url:
                                        thumbnail_url = candidate_url
                                        break
                                if thumbnail_url:
                                    break
                            except Exception:
                                continue
                except Exception:
                    pass
                
                # 상품 정보 수집 (제목이 없으면 스킵, 첫 상품은 디버깅 출력)
                if not title:
                    if idx == 0:
                        try:
                            print(f"\\n[디버깅] 상품 {{idx+1}} - 제목 추출 실패, outerHTML 일부")
                            target_elem = card_scope or item_elem
                            snippet = target_elem.get_attribute("outerHTML")[:400]
                            print(snippet)
                        except:
                            pass
                    continue
                
                product_data = {{
                    "title": title,
                    "original_price": original_price,
                    "displayed_price": displayed_price,
                    "product_link": product_link,
                    "thumbnail_url": thumbnail_url
                }}
                products.append(product_data)
                
                # 진행 상황 출력 (5개마다)
                if (idx + 1) % 5 == 0:
                    print(f"  진행: {{idx+1}}/{{min(len(product_items), max_products)}}개 수집 완료...")
                
                if len(products) >= max_products:
                    break
                
            except Exception as e:
                print(f"  상품 {{idx+1}} 정보 추출 오류: {{e}}")
                import traceback
                traceback.print_exc()
                continue
                    
except Exception as e:
    print(f"⚠ 크롤링 오류: {{e}}")
    import traceback
    traceback.print_exc()
finally:
    try:
        driver.quit()
    except:
        pass

# 결과 저장
result = {{
    "search_keyword": {SEARCH_KEYWORD!r},
    "search_url": search_url,
    "total_products": len(products),
    "products": products
}}

# JSON 파일로 저장
output_json = "naver_search_results.json"
with open(output_json, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print(f"\\n✅ 총 {{len(products)}}개 상품 정보 수집 완료")
print(f"✅ JSON 파일 저장 완료: {{output_json}}")

# 결과 미리보기
print(f"\\n📋 수집된 상품 미리보기 (처음 5개):")
for idx, product in enumerate(products[:5], 1):
    print(f"\\n  {{idx}}. {{product['title'][:60]}}...")
    if product.get('original_price'):
        print(f"     원가: {{product['original_price']}}")
    if product.get('displayed_price'):
        print(f"     할인가격: {{product['displayed_price']}}")
    print(f"     링크: {{product['product_link'][:70]}}..." if product['product_link'] else "     링크: 정보 없음")
"""

# 임시 파일에 스크립트 기록 후 실행
fd, path = tempfile.mkstemp(suffix="_crawl_naver.py")
os.close(fd)
with open(path, "w", encoding="utf-8") as f:
    f.write(crawl_script)

print("▶ 네이버 쇼핑 상품 검색 크롤링을 시작합니다...\n")

try:
    completed = subprocess.run(
        [sys.executable, path], capture_output=True, text=True, check=False
    )
    print(completed.stdout)
    if completed.returncode != 0:
        print("--- 프로세스가 에러로 종료되었습니다 (stderr) ---")
        print(completed.stderr)
finally:
    try:
        os.remove(path)
    except Exception:
        pass


🔍 검색 키워드: 가방
🔗 검색 URL: https://search.shopping.naver.com/ns/search?query=%EA%B0%80%EB%B0%A9
📦 최대 수집 개수: 30개

▶ 네이버 쇼핑 상품 검색 크롤링을 시작합니다...

브라우저 시작 중...
접속 중: https://search.shopping.naver.com/ns/search?query=%EA%B0%80%EB%B0%A9
네이버 메인 페이지 접속 중...

검색 페이지로 이동 중: https://search.shopping.naver.com/ns/search?query=%EA%B0%80%EB%B0%A9
페이지 로딩 대기 중...
상품 로딩 중...
발견된 상품 수: 86개 (셀렉터: div[class*='basicProductCardInformation'])
  진행: 25/30개 수집 완료...

✅ 총 15개 상품 정보 수집 완료
✅ JSON 파일 저장 완료: naver_search_results.json

📋 수집된 상품 미리보기 (처음 5개):

  1. 제이월드 신학기 16인치 롤링백팩 어린이 책가방 학원가방 어학연수 미니캐리어...
     원가: 80,000원
     할인가격: 59,000원
     링크: https://ader.naver.com/v1/imLnfDYhxdhf9ogD64ViDQYJij0dL_5HaivGWqLOseye...

  2. 오드비 비 마이 하트 경량 백팩 베이지 Beige B my Heart Lightweight Backpack ...
     원가: 168,000원
     할인가격: 151,200원
     링크: https://ader.naver.com/v1/r8Lm5RcqyzkQ_KqiUn6V4W5frTO15m6WAvPvSXTPDCHH...

  3. 오드비 비 마이 하트 경량 백팩 라이트블루 Light Blue B my Heart Lightweight Ba...
     원가: 168,000원
     할인가격: 151,200원
 